# GroupBy Fundamentals
Task 22 - Day 22

In [1]:
import pandas as pd

df = pd.DataFrame({
    "OrderID": range(1, 13),
    "Region": ["North", "South", "East", "North", "West", "South",
                "East", "North", "West", "South", "East", "West"],
    "Category": ["Furniture", "Technology", "Office", "Technology", "Furniture", "Office",
                  "Technology", "Office", "Technology", "Furniture", "Furniture", "Office"],
    "Sales": [1200, 2500, 400, 1800, 950, 600, 3100, 350, 2200, 1400, 1100, 500],
    "Quantity": [3, 5, 8, 4, 2, 10, 6, 7, 5, 4, 3, 9],
    "Profit": [200, 600, 80, 450, 90, 120, 800, 60, 520, 210, 150, 100],
})
df

,OrderID,Region,Category,Sales,Quantity,Profit
0,1,North,Furniture,1200,3,200
1,2,South,Technology,2500,5,600
2,3,East,Office,400,8,80
3,4,North,Technology,1800,4,450
4,5,West,Furniture,950,2,90
5,6,South,Office,600,10,120
6,7,East,Technology,3100,6,800
7,8,North,Office,350,7,60
8,9,West,Technology,2200,5,520
9,10,South,Furniture,1400,4,210


## Basic groupby with one aggregation

In [2]:
# 1. sum - total sales per region
print(df.groupby("Region")["Sales"].sum())

Region
East     4600
North    3350
South    4500
West     3650
Name: Sales, dtype: int64


In [3]:
# 2. mean - average sales per region
print(df.groupby("Region")["Sales"].mean())

Region
East     1533.333333
North    1116.666667
South    1500.000000
West     1216.666667
Name: Sales, dtype: float64


In [4]:
# 3. count - number of orders per category
print(df.groupby("Category")["OrderID"].count())

Category
Furniture     4
Office        4
Technology    4
Name: OrderID, dtype: int64


In [5]:
# 4. min - smallest sale per category
print(df.groupby("Category")["Sales"].min())

Category
Furniture      950
Office         350
Technology    1800
Name: Sales, dtype: int64


In [6]:
# 5. max - largest sale per category
print(df.groupby("Category")["Sales"].max())

Category
Furniture     1400
Office         600
Technology    3100
Name: Sales, dtype: int64


## Several aggregations at once

In [7]:
# sum, mean, count, min, max together
print(df.groupby("Category")["Sales"].agg(["sum", "mean", "count", "min", "max"]))

             sum    mean  count   min   max
Category                                   
Furniture   4650  1162.5      4   950  1400
Office      1850   462.5      4   350   600
Technology  9600  2400.0      4  1800  3100


In [8]:
# aggregating more than one column
print(df.groupby("Region")[["Sales", "Profit"]].sum())

        Sales  Profit
Region               
East     4600    1030
North    3350     710
South    4500     930
West     3650     710


In [9]:
# named aggregations - custom output column names
print(df.groupby("Region").agg(
    total_sales=("Sales", "sum"),
    avg_profit=("Profit", "mean"),
    orders=("OrderID", "count"),
))

        total_sales  avg_profit  orders
Region                                 
East           4600  343.333333       3
North          3350  236.666667       3
South          4500  310.000000       3
West           3650  236.666667       3


## Grouping by more than one column

In [10]:
print(df.groupby(["Region", "Category"])["Sales"].sum())

Region  Category  
East    Furniture     1100
        Office         400
        Technology    3100
North   Furniture     1200
        Office         350
        Technology    1800
South   Furniture     1400
        Office         600
        Technology    2500
West    Furniture      950
        Office         500
        Technology    2200
Name: Sales, dtype: int64


## Sorting and deriving from group results

In [11]:
# regions ranked by total sales
print(df.groupby("Region")["Sales"].sum().sort_values(ascending=False))

Region
East     4600
South    4500
West     3650
North    3350
Name: Sales, dtype: int64


In [12]:
# profit margin per category
margin = df.groupby("Category")[["Sales", "Profit"]].sum()
margin["Margin%"] = (margin["Profit"] / margin["Sales"] * 100).round(1)
print(margin)

            Sales  Profit  Margin%
Category                          
Furniture    4650     650     14.0
Office       1850     360     19.5
Technology   9600    2370     24.7


In [13]:
# average quantity per order, by category
print(df.groupby("Category")["Quantity"].mean())

Category
Furniture     3.0
Office        8.5
Technology    5.0
Name: Quantity, dtype: float64


## Interpretation of group-level results

East leads on total sales at 4,600, followed closely by South at 4,500, while North is lowest at 3,350. Every region has exactly three orders, so the gap comes from order value, not order volume. By category, Technology dominates with 9,600 in total sales and an average order of 2,400, more than five times the average Office order (462.5), even though each category has the same four orders. Technology also earns the best profit margin at 24.7%, with Office next at 19.5% and Furniture lowest at 14.0%, so Furniture brings in more revenue than Office (4,650 vs 1,850) but keeps a smaller share of it as profit. Office orders carry the highest average quantity (8.5 units per order) yet the lowest value, which points to many low-priced items per order.

## Interview questions

**What does groupby() do?**
It splits a DataFrame into groups based on the values in one or more columns, applies a function to each group independently, and combines the results into a new object. This split-apply-combine pattern is how you answer questions like "total sales per region" without writing a loop.

**What is aggregation?**
Reducing a group of values down to a single summary value, such as `sum`, `mean`, `count`, `min`, or `max`. After `groupby()`, an aggregation collapses each group into one row, and `agg()` lets you apply several aggregations at once or give the output columns custom names.

**How can you group by multiple columns?**
Pass a list of column names: `df.groupby(["Region", "Category"])["Sales"].sum()`. The result has a hierarchical (multi-level) index with one level per grouping column, giving one aggregated value for every unique combination, like the East + Technology total above.